# C. elegans — Temporal Neuron Roles: Partial Correlation Baseline

Prerequisite: the `correlation_partial` connectivity stage must already exist for the selected `RUN_ID`.
This optional analysis groups neurons by signed incoming/outgoing roles. Structural community detection remains available through installed Leiden; roles do not establish densely coupled modules.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused, failed, running, and missing checkpoints are reported explicitly.

Environment: select the project Python 3.14 `.venv` kernel. Install the locked packages with `uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks` from the repository root. The setup cell checks installed package versions; no upstream repository clone is required. Stored outputs may predate the backend migration; rerun the notebook to produce results with the current backend.

One role is the null summary. Increase the role count only with held-out evidence or an independently declared structural hypothesis.


In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]


if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Select the project Python 3.14 .venv kernel before running this notebook. "
        f"Current interpreter: {sys.executable}"
    )
_BACKEND_DISTRIBUTIONS = ("fastcgc", "tigramite", "scikit-learn", "kmedoids", "igraph", "leidenalg")
try:
    _BACKEND_VERSIONS = {name: version(name) for name in _BACKEND_DISTRIBUTIONS}
except PackageNotFoundError as error:
    raise RuntimeError(
        "The notebook kernel is missing a required installed backend. Run "
        "uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks "
        "from the repository root, then restart the project .venv kernel."
    ) from error
print("Notebook interpreter:", sys.executable)
print("Installed backends:", _BACKEND_VERSIONS)


In [ ]:
from effectome.community import CommunityConfig
from notebooks._shared import PROJECT_ROOT, make_run, print_notebook_header, print_stage_status, run_community

DATASET_ID = "c_elegans"
DATASET_OUTPUT_ID = "bundle-net-c-elegans"
RECORDING_ID = "bundle-net-worm-0"
RUN_ID = "reference"
METHOD = "correlation_partial"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
FORCE = False

COMMUNITY_CFG = CommunityConfig(
    name="temporal",
    resolution=1.0,
    symmetrize=False,
    use_absolute=False,
    seed=42,
    extra={
        "n_communities": 1,
        "feature_representation": "signed_in_out",
        "mode": "prospective",
        "temporal_penalty": 1.5,
        "n_runs": 5,
        "max_iter": 20,
        "init_window_count": 4,
        "online_learning_rate": 0.25,
    },
)

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_notebook_header(
    run,
    title=f"{METHOD} temporal neuron roles",
    expected_stages=["community"],
    force=FORCE,
)


In [ ]:
community = run_community(run, community_cfg=COMMUNITY_CFG, force=FORCE)
print(community.labels.shape)
print(community.method, getattr(community, "mode", None))
print_stage_status(run)
